# Notebook 38 - Capacity confound, train-validation duplicates, IoMT deep at twenty seeds

Three experiments the second hostile read demanded, with readings fixed in stage 2.

**D, the duplicate audit.** Exact-duplicate audit of CICIoT2023's standardised feature vectors between training and validation and within validation, per class, and the teachers and frozen shallow students evaluated at natural prior on all validation rows and on the rows absent from training, beside their locked test values. D1: the deep teacher's fine macro-F1 on validation rows absent from training is within 0.03 of its test value, in which case the validation-to-test gap is explained by duplication.

**P, the capacity control.** The deep CICIoT2023 scores are recomputed here on validation batches as in the original benchmark, and from the same removal orders two structures per method are calibrated: FLOP-matched (40% realised MAC reduction) and parameter-matched (40% parameter reduction). Twenty seeds each under minimal recovery, every model recalibrated. P0: the fresh FLOP-matched structures reproduce the NB37 ordering (magnitude lowest ground-truth HSR, V-C in the bottom two). P1: the parameter-matched structures preserve it; if they do, the sign reversal is not a capacity artefact.

**I, the second corpus at power.** CIC-IoMT-2024 deep, frozen NB33 structures, five methods by twenty seeds, minimal recovery, recalibrated. I1 reads the HSR ordering as channel-level, CICIoT2023-deep, or no significant pair.

**Stages.** 1 bootstrap, 2 pre-registration, 3 CICIoT2023 data and helpers, 4 duplicate audit, 5 scores, structures and the 200 CICIoT2023 runs (resumable), 6 the 100 IoMT runs (resumable), 7 analysis, 8 verdict and figures. GPU recommended; about three hours in total.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, hashlib, itertools, gc
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy import stats

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import enumerate_cnn1d_channel_groups, prune_cnn1d_channels, profile_forward_flops, count_parameters
from src.saber.leverage import magnitude_scores, gradient_saliency_scores, semantic_boundary_leverage

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; OUT = R / "38_capacity_duplicates_iomt20"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "P_capacity_duplicates_iomt20",
    "D_duplicate_audit": {
        "design": ("exact-duplicate audit of the standardised float32 feature vectors of CICIoT2023: fraction of validation rows whose "
                   "vector occurs in training, within-validation duplicate fraction, per-class rates; both teachers and the fifteen frozen "
                   "shallow students evaluated at natural prior on all validation rows and on validation rows absent from training; "
                   "CIC-IoMT-2024 reported for reference (its bridge removed exact duplicates before splitting)"),
        "D1": ("reading rule: if the deep CICIoT2023 teacher's fine macro-F1 on validation rows absent from training is within 0.03 of its "
               "locked test value (0.403), the validation-to-test gap is explained by train-validation duplication; otherwise it is not")},
    "P_parameter_matched": {
        "design": ("deep CICIoT2023 teacher; the five scores recomputed in this notebook on validation batches (as in the original benchmark); "
                   "for each method TWO structures from the same removal order: FLOP-matched (realised MAC reduction 40%) and PARAMETER-matched "
                   "(parameter reduction 40%), tolerance 1.5 points, minimum 8 channels per layer; 5 methods x 20 seeds x 2 matchings under "
                   "minimal recovery, every model recalibrated on the NB31 slice; analysis as in NB37"),
        "P1": ("at matched parameters, with 20 seeds, magnitude still has the lowest recalibrated ground-truth HSR and V-C is among the two "
               "highest, as at matched FLOPs; if this holds the sign reversal is not a capacity artefact; if magnitude loses its rank or V-C "
               "leaves the bottom two, capacity is part of the explanation and the paper says so"),
        "P0": "the FLOP-matched structures rebuilt from fresh scores reproduce the NB37 deep ordering (magnitude lowest HSR, V-C in the bottom two)"},
    "I_iomt_deep_twenty": {
        "design": "CIC-IoMT-2024 deep architecture, frozen NB33 structures, 5 methods x 20 seeds, minimal recovery, recalibrated; analysis as in NB37",
        "I1": ("descriptive with a fixed reading: (i) channel-level ordering (Taylor, Fisher or V-C lowest HSR), (ii) CICIoT2023-deep ordering "
               "(magnitude lowest), or (iii) no Holm-significant pair on HSR; whichever the data show is reported")},
    "seeds": [101, 211, 307, 401, 503, 613, 719, 823, 907, 1013, 1109, 1201, 1303, 1409, 1511, 1607, 1709, 1801, 1907, 2003],
    "no_test_access": True,
}
(OUT / "P_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2)); print(json.dumps(PREREG, indent=2))
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]; SEEDS20 = PREREG["seeds"]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; TOL = 0.015; TARGET = 0.40; MIN_W = 8


In [ ]:
# Stage 3 - CICIoT2023: data, deep teacher, helpers
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


DEEP = DeepCNN1D(N_CLASSES); DEEP.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
DEEP = DEEP.to(DEVICE).eval(); SHALLOW = SHALLOW_TEACHER.to(DEVICE).eval()
Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy(); Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


T_LOGITS_DEEP = forward_logits(DEEP, EX_X)


def audit_deep(model):
    lg = forward_logits(model, EX_X); a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS_DEEP, lg, EX_Y, robust_graph)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
            "fine_f1": float(a["fine_macro_f1"]), "awbir": float(aw), "hsr_balanced": float(a["hsr_balanced_soc"])}


def recalibrate_bn(model, loader, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(loader):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_subset_loader(dataset, n, seed):
    sub = torch.randperm(n, generator=torch.Generator().manual_seed(seed))[: int(n * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover_minimal(student, loader, weights):
    opt = torch.optim.Adam(student.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=weights)
    student.train()
    for xb, yb in loader:
        opt.zero_grad(); lossf(student(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return student.eval()


print("CICIoT2023 loaded | train", N_TRAIN, "| val", len(VAL_Y_ALL), "| eval rows", len(_idx))


In [ ]:
# Stage 4 - D: train-validation duplicate audit on CICIoT2023, and what it does to the validation-test gap
def row_hashes(X):
    return pd.util.hash_pandas_object(pd.DataFrame(X.numpy()), index=False).to_numpy()


h_tr, h_va = row_hashes(Xt), row_hashes(Xv)
in_train = np.isin(h_va, h_tr)
_, first_idx = np.unique(h_va, return_index=True); within_dup = np.ones(len(h_va), dtype=bool); within_dup[first_idx] = False
per_class = pd.DataFrame({"class": [CLASS_NAMES[c] for c in range(N_CLASSES)],
                          "val_rows": np.bincount(VAL_Y_ALL, minlength=N_CLASSES),
                          "val_rows_in_train": np.bincount(VAL_Y_ALL[in_train], minlength=N_CLASSES)})
per_class["rate"] = per_class.val_rows_in_train / per_class.val_rows.clip(lower=1)
per_class.to_csv(OUT / "D_duplicates_per_class.csv", index=False)
dup = {"val_rows": int(len(h_va)), "val_rows_with_vector_in_train": int(in_train.sum()), "fraction_val_in_train": float(in_train.mean()),
       "fraction_val_within_duplicates": float(within_dup.mean()), "train_rows": int(len(h_tr)), "fraction_train_within_duplicates": float(1 - len(np.unique(h_tr)) / len(h_tr))}
print(json.dumps(dup, indent=2)); print(per_class.sort_values("rate", ascending=False).head(10).to_string(index=False))

# teachers and frozen shallow students at natural prior: all validation rows vs rows absent from training
keep = ~in_train
Xv_all, Xv_keep = Xv.to(DEVICE), Xv[torch.from_numpy(keep)].to(DEVICE); Y_all, Y_keep = VAL_Y_ALL, VAL_Y_ALL[keep]
test = pd.read_csv(R / "21_one_shot_test/test_model_summary.csv"); ARCH_LABEL = {"shallow": "CNN1D-2block", "deep": "DeepCNN1D-4block"}


def fine_f1(model, X, Y):
    return float(full_model_audit(forward_logits(model, X), Y, taxonomy, DEFAULT_COST_PROFILES)["fine_macro_f1"])


rows = []
for arch, m in [("shallow", SHALLOW), ("deep", DEEP)]:
    t = test[(test.variant.astype(str) == "teacher") & (test.architecture == ARCH_LABEL[arch])].fine_macro_f1.iloc[0]
    rows.append({"model": f"{arch} teacher", "val_all": fine_f1(m, Xv_all, Y_all), "val_not_in_train": fine_f1(m, Xv_keep, Y_keep), "test": float(t)})
reg = pd.read_csv(R / "17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv")
for r in reg.itertuples():
    rm = pd.read_csv(R / f"17b_calibrated_checkpoint_freeze/{r.method}_r{int(round(float(r.target_flops) * 100))}cal_removed_groups.csv")
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(SHALLOW, pm, EXAMPLE_INPUT, minimum_remaining_per_layer=int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]))
    st.load_state_dict(torch.load(REPO / str(r.checkpoint), map_location="cpu", weights_only=False)["state_dict"]); st = st.to(DEVICE).eval()
    t = test[(test.architecture == ARCH_LABEL["shallow"]) & (test.method == r.method) & np.isclose(test.target_flops, float(r.target_flops))].fine_macro_f1.iloc[0]
    rows.append({"model": f"shallow {r.method} {float(r.target_flops):.2f}", "val_all": fine_f1(st, Xv_all, Y_all), "val_not_in_train": fine_f1(st, Xv_keep, Y_keep), "test": float(t)})
gap = pd.DataFrame(rows); gap["gap_all_minus_test"] = gap.val_all - gap.test; gap["gap_dedup_minus_test"] = gap.val_not_in_train - gap.test
gap.to_csv(OUT / "D_validation_test_gap.csv", index=False); print(gap.round(4).to_string(index=False))
deep_row = gap[gap.model == "deep teacher"].iloc[0]
D1 = bool(abs(deep_row.val_not_in_train - deep_row.test) <= 0.03)
print("\nD1 (deep teacher's fine macro-F1 on validation rows absent from training within 0.03 of test):", D1)
del Xv_keep; gc.collect()


In [ ]:
# Stage 5 - P: deep scores recomputed, FLOP-matched and parameter-matched structures, 20 seeds each
groups = enumerate_cnn1d_channel_groups(DEEP, EXAMPLE_INPUT)
mag = magnitude_scores(DEEP, groups)
grad = gradient_saliency_scores(DEEP, VAL_LOADER, groups, device=DEVICE, max_batches=40, criterion=nn.CrossEntropyLoss(weight=CLASS_W))
sbl_t, _ = semantic_boundary_leverage(DEEP, VAL_LOADER, groups, robust_graph, device=DEVICE, max_samples_per_class=256,
                                      edge_weight_column="robust_weight", normalize_by_group_size=0.0, normalize_by_flops=0.0)
for t in (groups, mag, grad, sbl_t): t["group_id"] = t["group_id"].astype(str)
S = (groups[["group_id", "module_path", "channel_index"]].merge(mag, on="group_id", validate="1:1")
     .merge(grad[["group_id", "taylor", "fisher"]], on="group_id", validate="1:1").merge(sbl_t[["group_id", "sbl_raw"]], on="group_id", validate="1:1"))
S["saber_v2"] = S.groupby("module_path")["sbl_raw"].rank(pct=True) * S["module_path"].map(S.groupby("module_path")["fisher"].mean())
S["random"] = np.random.default_rng(1).random(len(S)); S.to_csv(OUT / "P_deep_scores.csv", index=False)
assert len(S) == len(groups) == 576


def removal_order(column):
    left = {p: int((S["module_path"] == p).sum()) for p in S["module_path"].unique()}; seq = []
    for r in S.sort_values(column, ascending=True).itertuples():
        if left[r.module_path] - 1 < MIN_W: continue
        left[r.module_path] -= 1; seq.append((r.module_path, int(r.channel_index)))
    return seq


def prune_prefix(seq, k):
    pm = {}
    for p, c in seq[:k]: pm.setdefault(p, []).append(c)
    st, _ = prune_cnn1d_channels(DEEP, {p: sorted(cs) for p, cs in pm.items()}, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W); return st.to(DEVICE)


M_DENSE = profile_forward_flops(DEEP, EXAMPLE_INPUT)["flops_per_item"]; P_DENSE = count_parameters(DEEP)
flop_red = lambda st: 1 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / M_DENSE
param_red = lambda st: 1 - count_parameters(st) / P_DENSE
STRUCT = OUT / "P_structures.csv"; struct_rows = []
ORDERS = {m: removal_order(m) for m in METHODS}
for matching, measure in [("flops", flop_red), ("params", param_red)]:
    for method in METHODS:
        seq = ORDERS[method]; lo, hi = 1, len(seq)
        while lo < hi:
            mid = (lo + hi) // 2
            if measure(prune_prefix(seq, mid)) >= TARGET: hi = mid
            else: lo = mid + 1
        st = prune_prefix(seq, lo)
        pd.DataFrame([{"module_path": p, "channel_index": c} for p, c in seq[:lo]]).to_csv(OUT / f"P_deep_{method}_{matching}_removed_groups.csv", index=False)
        struct_rows.append({"matching": matching, "method": method, "k": lo, "flop_reduction": float(flop_red(st)), "param_reduction": float(param_red(st)),
                            "parameters": int(count_parameters(st)), "within_tolerance": bool(abs(measure(st) - TARGET) <= TOL), **{f"raw_{k}": v for k, v in audit_deep(st).items()}})
        print(f"{matching:6s} {method:9s}: k={lo} flops-red={struct_rows[-1]['flop_reduction']:.3f} params-red={struct_rows[-1]['param_reduction']:.3f} params={struct_rows[-1]['parameters']}")
pd.DataFrame(struct_rows).to_csv(STRUCT, index=False)
assert all(r["within_tolerance"] for r in struct_rows), "a structure is out of tolerance"

RUNS = OUT / "P_runs.csv"; rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["matching"], r["method"], r["seed"]) for r in rows}
for matching in ["flops", "params"]:
    for method in METHODS:
        rm = pd.read_csv(OUT / f"P_deep_{method}_{matching}_removed_groups.csv")
        pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
        for seed in SEEDS20:
            if (matching, method, seed) in done: continue
            torch.manual_seed(seed); np.random.seed(seed)
            st, _ = prune_cnn1d_channels(DEEP, pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W); st = st.to(DEVICE)
            raw = audit_deep(st); st = recover_minimal(st, make_subset_loader(TRAIN_LOADER.dataset, N_TRAIN, seed), CLASS_W)
            tr = audit_deep(st); recalibrate_bn(st, CAL_LOADER); rc = audit_deep(st)
            row = {"matching": matching, "method": method, "seed": seed, "parameters": int(count_parameters(st)), "raw_awbir": raw["awbir"], "raw_hsr_balanced": raw["hsr_balanced"]}
            row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); rows.append(row)
            pd.DataFrame(rows).to_csv(RUNS, index=False)
            print(f"{matching:6s} {method:9s} s{seed:4d}: HSR recal={rc['hsr_balanced']:.4f} awbir recal={rc['awbir']:.4f}")
print("P runs:", len(rows))


In [ ]:
# Stage 6 - I: CIC-IoMT-2024 deep, frozen NB33 structures, 20 seeds
from src.saber.bridge_iomt import build_bridge
I_TRAIN, I_VAL, I_CLASSES, i_tax, I_MAN = build_bridge(REPO / "data/iomt_bridge")
I_N = len(I_CLASSES); i_graph = pd.read_csv(R / "32_iomt_bridge/asvg_edges_robust.csv")


class IoMTDeep(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


_p = torch.load(REPO / "models/iomt/deep_teacher_seed0.pt", map_location="cpu", weights_only=False)
I_DEEP = IoMTDeep(I_N); I_DEEP.load_state_dict(_p["state_dict"]); I_DEEP = I_DEEP.to(DEVICE).eval(); I_ALPHA = float(_p["alpha"])
iXv, iYv = I_VAL.dataset.tensors; iY = iYv.numpy(); _r = np.random.default_rng(0)
_ii = np.concatenate([_r.permutation(np.where(iY == c)[0])[:4000] for c in range(I_N) if (iY == c).sum() > 0]); _ii = np.random.default_rng(12345).permutation(_ii)
I_EX_X = iXv[_ii].to(DEVICE); I_EX_Y = iY[_ii]; I_EXAMPLE = iXv[:8].float().to(DEVICE)
iXt, iYt = I_TRAIN.dataset.tensors; I_NT = len(iXt)
_c = np.bincount(iYt.numpy(), minlength=I_N); _w = np.zeros(I_N); _w[_c > 0] = 1 / np.power(_c[_c > 0], I_ALPHA); _w[_c > 0] /= _w[_c > 0].mean()
I_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_ci = torch.randperm(I_NT, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
I_CAL = torch.utils.data.DataLoader(torch.utils.data.Subset(I_TRAIN.dataset, _ci.tolist()), batch_size=1024, shuffle=False)
I_T_LOGITS = forward_logits(I_DEEP, I_EX_X)


def audit_iomt(model):
    lg = forward_logits(model, I_EX_X); a = full_model_audit(lg, I_EX_Y, i_tax, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(I_T_LOGITS, lg, I_EX_Y, i_graph, weight_column="robust_weight")
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
            "fine_f1": float(a["fine_macro_f1"]), "awbir": float(aw), "hsr_balanced": float(a["hsr_balanced_soc"])}


IRUNS = OUT / "I_runs.csv"; rows = pd.read_csv(IRUNS).to_dict("records") if IRUNS.exists() else []
done = {(r["method"], r["seed"]) for r in rows}
for method in METHODS:
    rm = pd.read_csv(R / f"33_iomt_scores_structures/deep_{method}_r40_removed_groups.csv")
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    for seed in SEEDS20:
        if (method, seed) in done: continue
        torch.manual_seed(seed); np.random.seed(seed)
        st, _ = prune_cnn1d_channels(I_DEEP, pm, I_EXAMPLE, minimum_remaining_per_layer=MIN_W); st = st.to(DEVICE)
        raw = audit_iomt(st); st = recover_minimal(st, make_subset_loader(I_TRAIN.dataset, I_NT, seed), I_W)
        tr = audit_iomt(st); recalibrate_bn(st, I_CAL); rc = audit_iomt(st)
        row = {"method": method, "seed": seed, "parameters": int(count_parameters(st)), "raw_awbir": raw["awbir"], "raw_hsr_balanced": raw["hsr_balanced"]}
        row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); rows.append(row)
        pd.DataFrame(rows).to_csv(IRUNS, index=False)
        print(f"iomt deep {method:9s} s{seed:4d}: HSR recal={rc['hsr_balanced']:.4f} awbir recal={rc['awbir']:.4f}")
print("I runs:", len(rows))


In [ ]:
# Stage 7 - analysis (shared with NB37), for the two CICIoT2023 matchings and the IoMT deep cell
rng = np.random.default_rng(0)


def shares(x):
    grand = x.mean(); ss_tot = ((x - grand) ** 2).sum()
    return float(x.shape[0] * ((x.mean(axis=0) - grand) ** 2).sum() / ss_tot), float(x.shape[1] * ((x.mean(axis=1) - grand) ** 2).sum() / ss_tot)


def perm_p(x, B=10000):
    m_obs, s_obs = shares(x); s_null, m_null = [], []
    for _ in range(B):
        xs = x.copy()
        for j in range(xs.shape[1]): xs[:, j] = rng.permutation(xs[:, j])
        s_null.append(shares(xs)[1]); xm = x.copy()
        for i in range(xm.shape[0]): xm[i, :] = rng.permutation(xm[i, :])
        m_null.append(shares(xm)[0])
    return float(np.mean(np.array(m_null) >= m_obs)), float(np.mean(np.array(s_null) >= s_obs))


def contrasts(piv):
    out = []; n = len(piv); tcrit = stats.t.ppf(0.975, n - 1)
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).values; m = d.mean(); se = d.std(ddof=1) / np.sqrt(n)
        out.append({"contrast": f"{a}-{b}", "mean": float(m), "half_width": float(tcrit * se), "p": float(stats.ttest_rel(piv[a], piv[b]).pvalue)})
    ps = sorted([(o["p"], i) for i, o in enumerate(out)]); running = 0.0; m_ = len(out)
    for k, (p, i) in enumerate(ps):
        running = max(running, min(1.0, p * (m_ - k))); out[i]["p_holm"] = running; out[i]["significant_holm"] = bool(running < 0.05)
    return out


def analyse(sub, label):
    cell = {}; raw = sub.groupby("method")[["raw_awbir", "raw_hsr_balanced"]].first()
    for metric in ["hsr_balanced", "awbir", "b2a"]:
        piv = sub.pivot_table(index="seed", columns="method", values=f"recal_{metric}")[METHODS]; p_m, p_s = perm_p(piv.values); cs = contrasts(piv)
        means = {m: float(v) for m, v in piv.mean().items()}; order = sorted(means, key=means.get)
        cell[metric] = {"means": means, "order_low_to_high": order, "p_method": p_m, "p_seed": p_s, "n_holm_significant": int(sum(c["significant_holm"] for c in cs)),
                        "max_abs_mean": float(max(abs(c["mean"]) for c in cs)), "mde": float(np.median([c["half_width"] for c in cs])),
                        "rank_rho": float(np.nanmean([piv.rank(axis=1).loc[a].corr(piv.rank(axis=1).loc[b], method="spearman") for a, b in itertools.combinations(piv.index, 2)]))}
        rawcol = f"raw_{metric}" if f"raw_{metric}" in raw.columns else None
        if rawcol:
            rs = raw[rawcol].max() - raw[rawcol].min(); cell[metric]["rei"] = float(1 - (piv.max(axis=1) - piv.min(axis=1)).mean() / rs) if rs > 0 else float("nan")
    cell["parameters_by_method"] = {m: int(sub[sub.method == m].parameters.iloc[0]) for m in METHODS}
    h = cell["hsr_balanced"]
    print(f"{label}: HSR order {h['order_low_to_high']} | means {dict((m, round(v, 4)) for m, v in h['means'].items())} | p_method {h['p_method']:.4f} | Holm {h['n_holm_significant']}/10 | rho {h['rank_rho']:.2f} | params {cell['parameters_by_method']}")
    return cell


P = pd.read_csv(OUT / "P_runs.csv"); I = pd.read_csv(OUT / "I_runs.csv")
analysis = {"ciciot_deep_flops_matched": analyse(P[P.matching == "flops"], "CICIoT2023 deep, FLOP-matched"),
            "ciciot_deep_params_matched": analyse(P[P.matching == "params"], "CICIoT2023 deep, parameter-matched"),
            "iomt_deep_flops_matched": analyse(I, "IoMT deep, FLOP-matched")}
json.dump(analysis, open(OUT / "P_I_analysis.json", "w"), indent=2)


In [ ]:
# Stage 8 - verdict and figures
analysis = json.load(open(OUT / "P_I_analysis.json")); gap = pd.read_csv(OUT / "D_validation_test_gap.csv")
def reading(cell):
    o = cell["hsr_balanced"]["order_low_to_high"]; sig = cell["hsr_balanced"]["n_holm_significant"]
    if sig == 0: return "no_significant_pair"
    if o[0] == "magnitude": return "ciciot_deep_ordering_magnitude_lowest"
    if o[0] in ("taylor", "fisher", "saber_v2"): return "channel_level_ordering"
    return "other"
P0 = (analysis["ciciot_deep_flops_matched"]["hsr_balanced"]["order_low_to_high"][0] == "magnitude") and ("saber_v2" in analysis["ciciot_deep_flops_matched"]["hsr_balanced"]["order_low_to_high"][-2:])
P1 = (analysis["ciciot_deep_params_matched"]["hsr_balanced"]["order_low_to_high"][0] == "magnitude") and ("saber_v2" in analysis["ciciot_deep_params_matched"]["hsr_balanced"]["order_low_to_high"][-2:])
deep_row = gap[gap.model == "deep teacher"].iloc[0]; D1 = bool(abs(deep_row.val_not_in_train - deep_row.test) <= 0.03)
per_class = pd.read_csv(OUT / "D_duplicates_per_class.csv")
verdict = {"arm": "P_capacity_duplicates_iomt20",
           "D1_gap_explained_by_train_val_duplication": D1,
           "D_deep_teacher": {k: float(deep_row[k]) for k in ["val_all", "val_not_in_train", "test"]},
           "D_fraction_val_in_train": float(per_class.val_rows_in_train.sum() / per_class.val_rows.sum()),
           "D_student_gap_mean": {"all_minus_test": float(gap[gap.model.str.startswith("shallow ")].gap_all_minus_test.mean()), "dedup_minus_test": float(gap[gap.model.str.startswith("shallow ")].gap_dedup_minus_test.mean())},
           "P0_fresh_flop_matched_reproduces_NB37_ordering": bool(P0), "P1_parameter_matched_preserves_ordering": bool(P1),
           "I1_iomt_deep_reading": reading(analysis["iomt_deep_flops_matched"]),
           "cells": {k: {"hsr": {kk: v["hsr_balanced"][kk] for kk in ["order_low_to_high", "means", "p_method", "n_holm_significant", "max_abs_mean", "mde", "rank_rho"]},
                         "awbir_order": v["awbir"]["order_low_to_high"], "awbir_holm": v["awbir"]["n_holm_significant"], "parameters": v["parameters_by_method"]} for k, v in analysis.items()},
           "prereg": json.load(open(OUT / "P_PREREGISTRATION.json"))}
(OUT / "P_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "cells")}, indent=2))
for k, v in verdict["cells"].items(): print(k, v["hsr"]["order_low_to_high"], {m: round(x, 4) for m, x in v["hsr"]["means"].items()}, "params", v["parameters"])

P = pd.read_csv(OUT / "P_runs.csv"); I = pd.read_csv(OUT / "I_runs.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (df, title) in zip(axes, [(P[P.matching == "flops"], "CICIoT2023 deep, FLOP-matched"), (P[P.matching == "params"], "CICIoT2023 deep, parameter-matched"), (I, "IoMT deep, FLOP-matched")]):
    ax.boxplot([df[df.method == m].recal_hsr_balanced.values for m in METHODS], showmeans=True); ax.set_xticks(range(1, 6)); ax.set_xticklabels(METHODS, fontsize=8)
    ax.set_title(title + " (20 seeds)"); ax.set_ylabel("ground-truth HSR, recalibrated")
fig.tight_layout(); fig.savefig(OUT / "P_I_hsr.png", dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(7, 3.2)); g = gap.set_index("model")
ax.plot(g.val_all.values, marker="o", label="validation, all rows"); ax.plot(g.val_not_in_train.values, marker="s", label="validation, rows absent from training"); ax.plot(g.test.values, marker="^", label="locked test")
ax.set_xticks(range(len(g))); ax.set_xticklabels(g.index, rotation=60, ha="right", fontsize=6); ax.set_ylabel("fine macro-F1"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "D_gap.png", dpi=200); plt.show(); print("written ->", OUT)


In [ ]:
# Stage 9 - addendum: the frozen criterion (L2 magnitude, as NB20 computed it), FLOP-matched check and parameter-matched at 20 seeds
conv_paths = sorted(S["module_path"].unique())
mods = dict(DEEP.named_modules())
S["magnitude_l2"] = [float(mods[p].weight[c].detach().pow(2).sum().sqrt().cpu()) for p, c in zip(S["module_path"], S["channel_index"])]
ORDERS["magnitude_l2"] = removal_order("magnitude_l2")
# does the L2 order at 40% FLOPs reproduce the frozen 20b magnitude structure exactly?
seq = ORDERS["magnitude_l2"]; lo, hi = 1, len(seq)
while lo < hi:
    mid = (lo + hi) // 2
    if flop_red(prune_prefix(seq, mid)) >= TARGET: hi = mid
    else: lo = mid + 1
frozen = pd.read_csv(R / "20b_depth_checkpoint_freeze/magnitude_minimal_r40_removed_groups.csv")
fresh_l2 = set((p, c) for p, c in seq[:lo]); frozen_set = set(zip(frozen.module_path.astype(str), frozen.channel_index.astype(int)))
print(f"L2 magnitude FLOP-matched: k={lo} params={count_parameters(prune_prefix(seq, lo))} | identical to frozen 20b structure: {fresh_l2 == frozen_set} "
      f"(overlap {len(fresh_l2 & frozen_set)} of {len(frozen_set)})")
# parameter-matched L2 magnitude structure
lo, hi = 1, len(seq)
while lo < hi:
    mid = (lo + hi) // 2
    if param_red(prune_prefix(seq, mid)) >= TARGET: hi = mid
    else: lo = mid + 1
st = prune_prefix(seq, lo)
print(f"L2 magnitude parameter-matched: k={lo} flops-red={flop_red(st):.3f} params-red={param_red(st):.3f} params={count_parameters(st)}")
pd.DataFrame([{"module_path": p, "channel_index": c} for p, c in seq[:lo]]).to_csv(OUT / "P_deep_magnitude_l2_params_removed_groups.csv", index=False)
pm = {p: sorted(c for pp, c in seq[:lo] if pp == p) for p in conv_paths if any(pp == p for pp, _ in seq[:lo])}
RUNS9 = OUT / "P_runs_magnitude_l2.csv"; rows9 = pd.read_csv(RUNS9).to_dict("records") if RUNS9.exists() else []
done9 = {r["seed"] for r in rows9}
for seed in SEEDS20:
    if seed in done9: continue
    torch.manual_seed(seed); np.random.seed(seed)
    st, _ = prune_cnn1d_channels(DEEP, pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W); st = st.to(DEVICE)
    raw = audit_deep(st); st = recover_minimal(st, make_subset_loader(TRAIN_LOADER.dataset, N_TRAIN, seed), CLASS_W)
    tr = audit_deep(st); recalibrate_bn(st, CAL_LOADER); rc = audit_deep(st)
    row = {"matching": "params", "method": "magnitude_l2", "seed": seed, "parameters": int(count_parameters(st)), "raw_awbir": raw["awbir"], "raw_hsr_balanced": raw["hsr_balanced"]}
    row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); rows9.append(row)
    pd.DataFrame(rows9).to_csv(RUNS9, index=False)
    print(f"params magnitude_l2 s{seed:4d}: HSR recal={rc['hsr_balanced']:.4f} awbir recal={rc['awbir']:.4f}")
r9 = pd.DataFrame(rows9)
P = pd.read_csv(OUT / "P_runs.csv"); pm_ = P[P.matching == "params"]
summary = {"magnitude_l2_params_matched_hsr_mean": float(r9.recal_hsr_balanced.mean()),
           "magnitude_l1_params_matched_hsr_mean": float(pm_[pm_.method == "magnitude"].recal_hsr_balanced.mean()),
           "magnitude_l1_flops_matched_hsr_mean": float(P[(P.matching == "flops") & (P.method == "magnitude")].recal_hsr_balanced.mean()),
           "magnitude_l2_flops_matched_hsr_mean_NB37": 0.1649,
           "paired_t_l2_vs_l1_params_matched": float(stats.ttest_rel(r9.sort_values("seed").recal_hsr_balanced.values, pm_[pm_.method == "magnitude"].sort_values("seed").recal_hsr_balanced.values).pvalue),
           "l2_params_vs_best_other": {m: float(stats.ttest_rel(r9.sort_values("seed").recal_hsr_balanced.values, pm_[pm_.method == m].sort_values("seed").recal_hsr_balanced.values).pvalue) for m in ["taylor", "fisher", "saber_v2", "random"]},
           "note": "frozen 20b structures used L2 magnitude, abs-of-summed-product Taylor and a simplified leverage; stage 5 used the library's L1 magnitude, sum-of-abs Taylor and full leverage; V-C and random structures are identical under both"}
(OUT / "P_magnitude_l2_addendum.json").write_text(json.dumps(summary, indent=2)); print(json.dumps(summary, indent=2))

In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cat ../.gitconfig > /root/.gitconfig
cat ../.git-credentials > /root/.git-credentials && chmod 600 /root/.git-credentials
python3 - <<'EOF'
import json
nb = json.load(open("notebooks/38_capacity_duplicates_iomt20.ipynb"))
for c in nb["cells"]:
    if c.get("cell_type") == "code":
        c["outputs"] = []; c["execution_count"] = None
json.dump(nb, open("/tmp/stripped.ipynb", "w"), ensure_ascii=False, indent=1)
EOF
blob=$(git hash-object -w /tmp/stripped.ipynb)
git update-index --add --cacheinfo 100644,$blob,notebooks/38_capacity_duplicates_iomt20.ipynb
git add results/saber/38_capacity_duplicates_iomt20
git commit -m "NB38 results: capacity control, duplicate audit, IoMT deep at 20 seeds, plus a stage-9 addendum added AFTER stage 5's result was seen. Stage 5's pre-registration said the deep scores were recomputed 'as in the original benchmark'; that was inaccurate: the frozen 20b structures used L2 magnitude, abs-of-summed-product Taylor and a simplified leverage (NB20), while stage 5 used the library's L1 magnitude, sum-of-abs Taylor and full leverage. V-C and random structures are identical under both (same parameter counts to the unit); Taylor and Fisher differ slightly and land within 0.001 HSR of NB37. Stage 9 confirms the diagnosis: the L2 magnitude order at 40% FLOPs reproduces the frozen 20b structure exactly (135 of 135 groups). P0 is therefore not a like-for-like test and is not counted as failed. Magnitude at depth, recalibrated HSR, 20 seeds: L2 FLOP-matched 0.165 (NB37), L1 FLOP-matched 0.177, L1 parameter-matched 0.170, L2 parameter-matched 0.174 (tied with Taylor p=0.88; better than V-C and random p<1e-6): never worse than Taylor/Fisher, always better than V-C and random, best in one realisation of four. P1 HELD: at matched parameters (~109k for all five, V-C retaining the most FLOPs) V-C is second-worst (0.184), as at matched FLOPs (0.183), so the capacity confound is ruled out for the V-C result. I1: IoMT deep at 20 seeds follows the channel-level ordering (Fisher 0.131 < Taylor 0.134 < V-C 0.137 < random 0.139 < magnitude 0.145, 5/10 Holm). D1: zero exact duplicates between training and validation, within validation, or within training; the 0.09-0.12 validation-to-test drop is not leakage and remains unexplained"
git push origin saber-ids-method
git log --oneline -1